In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/akashraj77/signal-array/new_signal_values.txt
/kaggle/input/datasets/akashraj77/dataset-1/set2_3_features.csv


In [2]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

2026-04-10 12:05:33.745958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775822733.768013     223 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775822733.775590     223 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775822733.793171     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775822733.793189     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775822733.793192     223 computation_placer.cc:177] computation placer alr

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
tf.debugging.set_log_device_placement(True)

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

In [5]:
# Load your dataset
df = pd.read_csv("/kaggle/input/datasets/akashraj77/dataset-1/set2_3_features.csv")

# Separate features and target
X = df.drop(columns=["RUL(min)"])   # Features
y = df["RUL(min)"]                  # Target

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Step 1: Random Forest for feature transformation
rf = RandomForestRegressor(n_estimators=300, random_state=42)

# Train RF
rf.fit(X_train, y_train)

# Transform features using RF (leaf indices)
X_train_rf = rf.apply(X_train)
X_test_rf = rf.apply(X_test)

In [8]:
scaler = StandardScaler()

# Scale RF output
X_train_rf = scaler.fit_transform(X_train_rf)
X_test_rf = scaler.transform(X_test_rf)

In [9]:
param_grid = {
    "C": [10, 50, 100],
    "gamma": ["scale", 0.01, 0.001],
    "epsilon": [0.1, 1, 10]
}

grid = GridSearchCV(
    SVR(kernel="rbf"),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train_rf, y_train)

svm = grid.best_estimator_

In [10]:
y_pred = svm.predict(X_test_rf)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 7212.65973239391
R2 Score: 0.670041346496462


In [11]:
def extract_features(signal):
    signal = np.array(signal)
    features = {}

    # basic
    mean = np.mean(signal)
    std = np.std(signal)
    rms = np.sqrt(np.mean(signal**2))
    peak = np.max(np.abs(signal))

    features["mean"] = mean
    features["std"] = std
    features["rms"] = rms
    features["max"] = np.max(signal)
    features["min"] = np.min(signal)
    features["peak_to_peak"] = np.ptp(signal)

    # statistical
    features["skewness"] = pd.Series(signal).skew()
    features["kurtosis"] = pd.Series(signal).kurt()

    # vibration specific
    mean_abs = np.mean(np.abs(signal))
    sqrt_mean = np.mean(np.sqrt(np.abs(signal)))

    features["crest_factor"] = peak / rms if rms != 0 else 0
    features["shape_factor"] = rms / mean_abs if mean_abs != 0 else 0
    features["impulse_factor"] = peak / mean_abs if mean_abs != 0 else 0
    features["clearance_factor"] = peak / (sqrt_mean**2) if sqrt_mean != 0 else 0

    # entropy
    hist, _ = np.histogram(signal, bins=50, density=True)
    hist = hist + 1e-12
    features["entropy"] = -np.sum(hist * np.log(hist))

    # frequency features
    fft_vals = np.abs(np.fft.fft(signal))
    fft_vals = fft_vals[:len(fft_vals)//2]

    features["fft_mean"] = np.mean(fft_vals)
    features["fft_std"] = np.std(fft_vals)
    features["fft_max"] = np.max(fft_vals)

    # energy
    features["energy"] = np.sum(signal**2)
    features["log_energy"] = np.sum(np.log(signal**2 + 1e-12))

    return pd.DataFrame([features])

In [12]:
def predict_rul_from_raw(signal):
    # Step 1: Extract features
    features = extract_features(signal)

    # Step 2: Align columns with training data
    features = features[X.columns]

    # Step 3: RF transformation
    features_rf = rf.apply(features)

    # Step 4: Scaling
    features_rf = scaler.transform(features_rf)

    # Step 5: Prediction
    rul = svm.predict(features_rf)

    return rul[0]

In [15]:
from pathlib import Path

signal_path = Path("/kaggle/input/datasets/akashraj77/test-signal")

if signal_path.is_dir():
    signal_files = sorted(
        p for p in signal_path.rglob("*")
        if p.is_file() and p.suffix.lower() in {".txt", ".csv", ".dat"}
    )
    if not signal_files:
        raise FileNotFoundError(f"No readable signal file found inside {signal_path}")
    signal_path = signal_files[0]

new_signal = np.loadtxt(signal_path, dtype=float, encoding="utf-8-sig")

predicted_rul = predict_rul_from_raw(new_signal)

print(f"Loaded {new_signal.size} samples from {signal_path.name}")
print("Predicted RUL in minutes:", predicted_rul)

Loaded 20480 samples from test_signal_2.txt
Predicted RUL in minutes: 3806.6550585117147


In [14]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df)

             Feature  Importance
13          fft_mean    0.708495
7           kurtosis    0.073850
1                std    0.071758
2                rms    0.021962
0               mean    0.018780
16            energy    0.018280
15           fft_max    0.015556
9       shape_factor    0.014514
6           skewness    0.012151
14           fft_std    0.009176
4                min    0.006862
3                max    0.006486
12           entropy    0.006445
5       peak_to_peak    0.004577
8       crest_factor    0.004008
11  clearance_factor    0.003697
10    impulse_factor    0.003403
